# 03 - Genie Agent: ask the stress results in natural language

Creates (or updates) the Genie space defined in `genie/space.json`, then asks the five
sample questions through the Genie Conversation API and prints the SQL Genie wrote and
its answer. The space covers the three gold tables and the committee approvals that
the app writes to Lakebase (read through the Lakebase catalog in Unity Catalog).

In [1]:
import json, logging, warnings, yaml
warnings.filterwarnings("ignore")
logging.getLogger("databricks.sdk").setLevel(logging.ERROR)
from databricks.sdk import WorkspaceClient

cfg = yaml.safe_load(open("../config.yml"))
w = WorkspaceClient(profile=cfg["profile"])
space = json.load(open("../genie/space.json"))
TITLE = "Stress testing: risk committee"
DESCRIPTION = ("Answers risk committee questions on market risk stress test results: stressed P&L by scenario, "
               "desk and risk factor, desk stress loss limits and breaches, AI commentary, and committee decisions.")

In [2]:
found = [s for s in (w.genie.list_spaces().spaces or []) if s.title == TITLE]
if found:
    space_id = found[0].space_id
    w.genie.update_space(space_id, serialized_space=json.dumps(space), description=DESCRIPTION,
                         warehouse_id=cfg["warehouse_id"])
else:
    space_id = w.genie.create_space(cfg["warehouse_id"], json.dumps(space), title=TITLE, description=DESCRIPTION,
                                    parent_path=f"/Workspace/Users/{w.current_user.me().user_name}").space_id
print("Genie space:", TITLE, "| tables:", [t["identifier"] for t in space["data_sources"]["tables"]])

Genie space: Stress testing: risk committee | tables: ['stress_demo.risk.gold_commentary', 'stress_demo.risk.gold_desk_stress', 'stress_demo.risk.gold_stress_pnl', 'stress_demo_lakebase.committee.approvals']


## Ask the sample questions

In [3]:
for q in [s["question"][0] for s in space["config"]["sample_questions"]]:
    msg = w.genie.start_conversation_and_wait(space_id, q)
    print("=" * 100, "\nQ:", q, "\nstatus:", msg.status.value)
    for a in msg.attachments or []:
        if a.query:
            print("SQL:", a.query.query.strip())
            res = w.genie.get_message_attachment_query_result(space_id, msg.conversation_id, msg.message_id,
                                                              a.attachment_id).statement_response
            cols = [c.name for c in res.manifest.schema.columns]
            print("RESULT:", cols)
            for row in (res.result.data_array or [])[:10]:
                print("  ", row)
        if a.text:
            print("ANSWER:", a.text.content.strip())

Q: Which desks breach their stress loss limit, and in which scenarios? 
status: COMPLETED
SQL: SELECT `scenario_name`, `desk`, ROUND(try_divide(-`pnl_jpy`,1e9), 1) AS `loss_jpy_bn`, ROUND(try_divide(`limit_jpy`,1e9), 1) AS `limit_jpy_bn`, `limit_utilization` FROM `stress_demo`.`risk`.`gold_desk_stress` WHERE `limit_breach` = TRUE AND `scenario_name` IS NOT NULL AND `desk` IS NOT NULL AND `limit_utilization` IS NOT NULL ORDER BY `limit_utilization` DESC


RESULT: ['scenario_name', 'desk', 'loss_jpy_bn', 'limit_jpy_bn', 'limit_utilization']
   ['Yen surges 15%', 'FX', '11.6', '8.0', '1.455']
   ['Credit spreads widen', 'Credit', '4.9', '4.0', '1.225']
ANSWER: **2 desks** breach their stress loss limits in the visible data: **FX** in **Yen surges 15%** with a **JPY 11.6bn** loss against an **JPY 8.0bn** limit, and **Credit** in **Credit spreads widen** with a **JPY 4.9bn** loss against an **JPY 4.0bn** limit. Among these breaches, **FX** is the larger overrun, using **145.5%** of its limit versus **122.5%** for **Credit**.


Q: What are the top 3 risk factors behind the loss in the yen surge scenario? 
status: COMPLETED
SQL: WITH ranked_factors AS (
  SELECT
    `risk_factor`,
    `desk`,
    ROUND(try_divide(`pnl_jpy`,1e9), 1) AS `pnl_jpy_bn`,
    RANK() OVER (ORDER BY `pnl_jpy` ASC) AS `loss_rank`
  FROM `stress_demo`.`risk`.`gold_stress_pnl`
  WHERE `scenario_name` = 'Yen surges 15%'
    AND `risk_factor` IS NOT NULL
    AND `pnl_jpy` IS NOT NULL
)
SELECT
  `risk_factor`,
  `desk`,
  `pnl_jpy_bn`
FROM ranked_factors
WHERE `loss_rank` <= 3
ORDER BY `loss_rank`, `risk_factor`, `desk`


RESULT: ['risk_factor', 'desk', 'pnl_jpy_bn']
   ['USDJPY', 'FX', '-7.2']
   ['EURJPY', 'FX', '-4.4']
   ['NIKKEI225', 'Equity', '-1.3']
ANSWER: The top 3 risk factors behind the loss in the **Yen surges 15%** scenario are:
- **USDJPY** (**FX**): **-7.2 JPY bn**
- **EURJPY** (**FX**): **-4.4 JPY bn**
- **NIKKEI225** (**Equity**): **-1.3 JPY bn**
Across these 3 results, **FX** is the main driver, accounting for the two largest losses.
ANSWER: Would you prefer to see the top 3 risk factors aggregated across all desks instead of broken down by each desk?


Q: How much headroom does the Equity desk have under the global equity sell-off? 
status: COMPLETED
SQL: SELECT ROUND(try_divide((`limit_jpy` + `pnl_jpy`),1e9), 1) AS `headroom_jpy_bn` FROM `stress_demo`.`risk`.`gold_desk_stress` WHERE `scenario_name` = 'Global equity sell-off' AND `desk` = 'Equity' AND `scenario_name` IS NOT NULL AND `desk` IS NOT NULL


RESULT: ['headroom_jpy_bn']
   ['0.0']
ANSWER: The **Equity** desk has **JPY 0.0bn** of headroom under the **Global equity sell-off** scenario. Based on the single result shown, the **Equity** desk is exactly at its limit in this scenario.


Q: Which scenarios have not been signed off by the committee yet? 
status: COMPLETED
SQL: WITH `latest_approvals` AS (
  SELECT
    `scenario_id`,
    `decision`,
    `decided_at`,
    ROW_NUMBER() OVER (PARTITION BY `scenario_id` ORDER BY `decided_at` DESC) AS `rn`
  FROM `stress_demo_lakebase`.`committee`.`approvals`
)
SELECT
  `gc`.`scenario_name`,
  `la`.`decision` AS `current_decision`,
  `la`.`decided_at`
FROM `stress_demo`.`risk`.`gold_commentary` AS `gc`
LEFT JOIN `latest_approvals` AS `la`
  ON `gc`.`scenario_id` = `la`.`scenario_id`
 AND `la`.`rn` = 1
WHERE `gc`.`scenario_name` IS NOT NULL
  AND (`la`.`decision` IS NULL OR `la`.`decision` <> 'SIGNED_OFF')
ORDER BY `gc`.`scenario_name` ASC


RESULT: ['scenario_name', 'current_decision', 'decided_at']
   ['Credit spreads widen', None, None]
   ['Global equity sell-off', None, None]
   ['Yen surges 15%', 'ACTION_REQUESTED', '2026-11-16T10:49:06.999Z']
ANSWER: There are **3 scenarios** that have **not** been signed off by the committee: **Credit spreads widen**, **Global equity sell-off**, and **Yen surges 15%**. The first two show **no recorded decision**, while **Yen surges 15%** is currently marked **ACTION_REQUESTED** as of **2026-11-16 10:49:06 UTC**, so none of the three are yet signed off.


Q: Rank the scenarios by total stressed loss. 
status: COMPLETED
SQL: WITH ranked AS (
  SELECT
    `scenario_name`,
    ROUND(try_divide(-`total_pnl_jpy`,1e9), 1) AS `total_stressed_loss_jpy_bn`,
    RANK() OVER (ORDER BY `total_pnl_jpy` ASC) AS `loss_rank`
  FROM `stress_demo`.`risk`.`gold_commentary`
  WHERE `scenario_name` IS NOT NULL
    AND `total_pnl_jpy` IS NOT NULL
)
SELECT
  `loss_rank`,
  `scenario_name`,
  `total_stressed_loss_jpy_bn`
FROM ranked
ORDER BY `loss_rank`, `scenario_name`


RESULT: ['loss_rank', 'scenario_name', 'total_stressed_loss_jpy_bn']
   ['1', 'Global equity sell-off', '14.3']
   ['2', 'Yen surges 15%', '13.7']
   ['3', 'Credit spreads widen', '9.1']
   ['4', 'BoJ hikes 100bp', '5.3']
ANSWER: Ranked by **total stressed loss**, the scenarios are:
- **1. Global equity sell-off** — **JPY 14.3bn**
- **2. Yen surges 15%** — **JPY 13.7bn**
- **3. Credit spreads widen** — **JPY 9.1bn**
- **4. BoJ hikes 100bp** — **JPY 5.3bn**
Across the 4 scenarios shown, **Global equity sell-off** is the largest loss and **BoJ hikes 100bp** is the smallest.
